# 18 — Entanglement

Give a name and a number to what CNOT manufactured in `01/17`: a two-qubit state that is perfectly definite as a whole yet whose parts are maximally uncertain. Define entanglement for pure states, measure it by the entropy of a part computed by hand, and then be careful — the same entropy on a *mixed* whole measures nothing: the correlated coins of `01/16` have one full bit of it and no entanglement at all. You will learn the test that works for mixtures too, and the second-basis measurement that tells a Bell pair from a shared coin.

**Prerequisites:** `01/08_hadamard_and_measurement_bases` ($H$ before reading = measuring $X$), `01/13_pure_vs_mixed` (mixed states, purity, the von Neumann entropy preview: entropy of the eigenvalue coin), `01/16_tensor_and_partial_trace` (partial trace by blocks; the correlated coins of §5; three ways to be mixed), `01/17_cnot_and_two_qubit_gates` (the Bell state from $H$ and CNOT; the amplitude-matrix determinant; corners are coherence, not a meter; $Z$-counts cannot tell Bell from coins).

**What you'll be able to do**
- State the definition: a pure two-qubit state is **entangled** when it is not a product — and show that its parts are then mixed, which no classical joint distribution allows.
- Compute the **entanglement entropy** $S(\rho_A)$ by hand from the eigenvalues of the part, prove $S(\rho_A) = S(\rho_B)$, and see it is unchanged by local gates.
- Say precisely when that number means something — *only* for a pure whole — and use the partial-transpose test for mixed two-qubit states (the coins pass, the Bell state fails).
- Tell the Bell pair from the coins with a second measurement basis: $\langle X\otimes X\rangle = 1$ versus $0$.

In `01/17` you watched CNOT turn $|+\rangle\otimes|0\rangle$ into $\tfrac1{\sqrt2}(|00\rangle + |11\rangle)$: the output was pure, not a product, and its parts were $I/2$. That combination — **pure whole, mixed parts** — is the thing this notebook is about. It is the resource the rest of the course keeps returning to, and it is also the easiest thing in quantum information to over-read, so we will earn every claim.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS
from qot_course.quantum.gates import IDENTITY, PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, ry, rz
from qot_course.quantum.density import density_matrix, maximally_mixed, purity, von_neumann_entropy
from qot_course.quantum.composite import tensor, bell_state, partial_trace, entanglement_entropy

rng = np.random.default_rng(0)   # seeds for the simulator run and the random local gates of section 2
viz.use_course_style()

## 1. The definition, and why no table can do this

A **pure** two-qubit state $|\Psi\rangle$ is a **product** if it can be written $|\varphi\rangle\otimes|\chi\rangle$, and **entangled** if it cannot. From `01/17` you own the test: write the four amplitudes as the $2\times2$ amplitude matrix $\Psi_{ab}$; the state is a product if and only if $\det\Psi = 0$. For the Bell state $|\Phi^+\rangle = \tfrac1{\sqrt2}(|00\rangle + |11\rangle)$, $\Psi = \tfrac1{\sqrt2}I$ and $\det\Psi = \tfrac12$: entangled.

Now the consequence, which is the heart of the matter. Take the partial trace by blocks (`01/16` §3): $\rho_{AB} = |\Phi^+\rangle\langle\Phi^+|$ has $\tfrac12$ at the four corners $(00,00), (00,11), (11,00), (11,11)$ and zeros elsewhere, so

$$
\rho_A = \mathrm{tr}_B\,\rho_{AB} = \begin{pmatrix} \mathrm{tr}\,[\rho]_{00} & \mathrm{tr}\,[\rho]_{01} \\ \mathrm{tr}\,[\rho]_{10} & \mathrm{tr}\,[\rho]_{11}\end{pmatrix}
= \begin{pmatrix} \tfrac12 & 0 \\ 0 & \tfrac12 \end{pmatrix} = \tfrac{I}{2}:
$$

the off-diagonal blocks $[\rho]_{01} = \tfrac12|0\rangle\langle1|$ and $[\rho]_{10} = \tfrac12|1\rangle\langle0|$ have trace $0$ — the corners vanish under the partial trace — and each diagonal block has trace $\tfrac12$. The whole has purity $1$; each part has purity $\tfrac12$, the most mixed a qubit can be. In general: **a pure state is entangled if and only if its parts are mixed** (a product has pure parts by `01/16`'s product rule; conversely, a pure state with a pure part is a product — exercise 2 proves it).

Why is this impossible classically? A classical "pure whole" is a joint table with *one* outcome of probability $1$ — a point mass $p(a,b) = \delta_{a a_0}\delta_{b b_0}$ — and its marginals are point masses too: a certain whole has certain parts. The only classical route to mixed marginals is a mixed whole — the coins of `01/16` §5, which are a *mixture* (purity $\tfrac12$) of the two point masses $|00\rangle$ and $|11\rangle$. The Bell state has the coins' marginals and the coins' $Z$-statistics with a **pure** whole. That is new. In entropy language (`01/13`, `02/01`): classically a part is never more uncertain than the whole, $H(A) \le H(AB)$ (proved in `02/01`) — the coins respect it ($1 \le 1$); the Bell state has $S(A) = 1$ bit and $S(AB) = 0$, and the inequality fails (`02/11` turns this into a negative conditional entropy).

In [ ]:
bell = bell_state()
labels = ["|00>", "|01>", "|10>", "|11>"]

def amplitude_matrix(psi):
    return np.asarray(psi).reshape(2, 2)             # Psi[a, b] = amplitude of |ab>   (01/17)

def trace_out_B(rho4):                                # replace each 2x2 block by its trace  (01/16)
    return np.array([[np.trace(rho4[2 * a: 2 * a + 2, 2 * a_: 2 * a_ + 2]) for a_ in range(2)] for a in range(2)])

def trace_out_A(rho4):                                # add the diagonal blocks  (01/16)
    return rho4[0:2, 0:2] + rho4[2:4, 2:4]

rho_bell = np.outer(bell, bell.conj())
print("det Psi (Bell) =", np.linalg.det(amplitude_matrix(bell)).real.round(3), " -> not a product")
print("purity of the whole:", round(purity(rho_bell), 3))
rA, rB = trace_out_B(rho_bell), trace_out_A(rho_bell)
print("rho_A by blocks =\n", np.round(rA.real, 3), "\n== I/2:", np.allclose(rA, maximally_mixed(2)), "  == partial_trace:", np.allclose(rA, partial_trace(rho_bell, keep=[0], dims=[2, 2])))
print("rho_B by blocks == I/2:", np.allclose(rB, maximally_mixed(2)))
print("purity of a part:", round(purity(rA), 3))

# the classical comparison: a certain whole has certain parts; the coins are a mixed whole
point_mass = density_matrix(tensor(KET_0, KET_1))                                     # the joint outcome (0, 1) for sure
coins = 0.5 * density_matrix(tensor(KET_0, KET_0)) + 0.5 * density_matrix(tensor(KET_1, KET_1))
for name, rho in [("point mass |01>", point_mass), ("coins (01/16)", coins), ("Bell", rho_bell)]:
    print(f"{name:16s} purity(whole) = {purity(rho):.2f}   purity(part A) = {purity(trace_out_B(rho)):.2f}")

**Read the output.** The determinant is $\tfrac12$ — not a product. The block trace of the Bell density matrix gives $I/2$ for A and for B, exactly as computed by hand, and matches `partial_trace`. The purity table is the whole story in three lines: the point mass is pure with pure parts (the classical "certain whole"); the coins have mixed parts but a mixed whole; the Bell state has **pure whole and mixed parts** — the row no classical table can produce. Entanglement, defined and seen.

## 2. A number for it: the entropy of a part, by hand

How *much* entangled? For a pure whole, the natural measure is how mixed the part is — and the right yardstick for "how mixed" is the one `01/13` previewed: the **von Neumann entropy**, the Shannon entropy of the eigenvalue coin,

$$
S(\rho_A) = -\sum_i \lambda_i \log_2 \lambda_i ,\qquad \lambda_i = \text{eigenvalues of } \rho_A .
$$

For $\rho_A = I/2$: eigenvalues $\tfrac12, \tfrac12$, $S = -2\cdot\tfrac12\log_2\tfrac12 = 1$ bit — the maximum for a qubit. For a pure part (a product whole): eigenvalues $1, 0$, $S = 0$. This $S(\rho_A)$ is the **entanglement entropy**.

Two facts you can check by hand with the amplitude matrix. First, $\rho_A = \Psi\Psi^\dagger$ and $\rho_B = \Psi^{\mathsf T}\overline{\Psi}$ — read the index formulas $(\rho_A)_{aa'} = \sum_b \Psi_{ab}\overline{\Psi_{a'b}}$, $(\rho_B)_{bb'} = \sum_a \Psi_{ab}\overline{\Psi_{ab'}}$ — and for a $2\times2$ $\Psi$ both $\rho_A$ and $\rho_B$ have trace $\|\Psi\|^2 = 1$ and determinant $|\det\Psi|^2$ — the same characteristic polynomial, hence the same eigenvalues — so **$S(\rho_A) = S(\rho_B)$**: both parts of a pure state are equally mixed. (A corollary of the Schmidt decomposition, Nielsen & Chuang §2.5, which holds in any dimension.) Second, the eigenvalues of $\Psi\Psi^\dagger$ for a $2\times2$ $\Psi$ satisfy $\lambda_1 + \lambda_2 = 1$ and $\lambda_1\lambda_2 = |\det\Psi|^2$ — so the determinant of `01/17` and the entropy of today are two readings of the same pair of numbers: $\lambda_{1,2} = \tfrac12\big(1 \pm \sqrt{1 - 4|\det\Psi|^2}\big)$.

Let's compute it by hand on the family $\cos\tfrac\theta2\,|00\rangle + \sin\tfrac\theta2\,|11\rangle$ — CNOT applied to $R_y(\theta)|0\rangle \otimes |0\rangle$, the output of `01/17` exercise 4 — whose part is $\rho_A = \mathrm{diag}(\cos^2\tfrac\theta2, \sin^2\tfrac\theta2)$, so $S = h(\cos^2\tfrac\theta2)$ with $h(p) = -p\log_2 p - (1-p)\log_2(1-p)$ the binary entropy (the Shannon entropy of a $p$-coin, `02/01`): $0$ at $\theta = 0$ (product $|00\rangle$), $1$ bit at $\theta = \pi/2$ (the Bell state), $0$ again at $\theta = \pi$ ($|11\rangle$). And we check that a **local** gate $U\otimes V$ — anything each party does alone — leaves $S(\rho_A)$ unchanged: $\rho_A \to U\rho_A U^\dagger$ keeps the eigenvalues. Entanglement is not something either party can make or unmake alone.

In [ ]:
def binary_entropy(p):
    q = np.clip(np.array(p, dtype=float), 1e-15, 1 - 1e-15)
    return -(q * np.log2(q) + (1 - q) * np.log2(1 - q))

def entropy_by_hand(rho):
    lam = np.linalg.eigvalsh(rho).real
    lam = lam[lam > 1e-12]
    return float(-(lam * np.log2(lam)).sum())

print("eigenvalues of I/2:", np.linalg.eigvalsh(rA).real.round(3), " S by hand =", round(entropy_by_hand(rA), 3), "bit",
      "  == von_neumann_entropy:", np.isclose(entropy_by_hand(rA), von_neumann_entropy(rA)),
      "  == entanglement_entropy(rho_bell):", np.isclose(entropy_by_hand(rA), entanglement_entropy(rho_bell, dims=[2, 2])))
prod = tensor(KET_PLUS, KET_0)
print("product |+>|0>: S(rho_A) =", round(abs(entanglement_entropy(density_matrix(prod), dims=[2, 2])), 3), "bit")

# S(rho_A) == S(rho_B) and the det link, on a random pure state (no symmetry to lean on)
psi = rng.normal(size=4) + 1j * rng.normal(size=4); psi /= np.linalg.norm(psi)
P = amplitude_matrix(psi)
rhoA, rhoB = P @ P.conj().T, P.T @ P.conj()
lamA, lamB = np.linalg.eigvalsh(rhoA).real, np.linalg.eigvalsh(rhoB).real
d = abs(np.linalg.det(P))
print("\nrandom pure state: eigenvalues of rho_A", lamA.round(4), " of rho_B", lamB.round(4), " equal:", np.allclose(lamA, lamB))
print("  lambda_1 + lambda_2 =", lamA.sum().round(4), "  lambda_1 lambda_2 =", np.prod(lamA).round(4), "  |det Psi|^2 =", (d**2).round(4))
print("  S(rho_A) =", round(entropy_by_hand(rhoA), 4), " S(rho_B) =", round(entropy_by_hand(rhoB), 4))

# local gates leave S unchanged
U, V = rz(rng.uniform(0, 2 * np.pi)) @ ry(rng.uniform(0, np.pi)), HADAMARD @ ry(rng.uniform(0, np.pi))
psi_local = tensor(U, V) @ psi
print("  after a random U (x) V:  S(rho_A) =", round(entanglement_entropy(density_matrix(psi_local), dims=[2, 2]), 4), " (unchanged)")

# the family cos(theta/2)|00> + sin(theta/2)|11>: entropy by hand vs the helper
thetas = np.linspace(0, np.pi, 181)
S_hand = binary_entropy(np.cos(thetas / 2) ** 2)
S_lib = np.array([entanglement_entropy(density_matrix(np.array([np.cos(t / 2), 0, 0, np.sin(t / 2)], dtype=complex)), dims=[2, 2]) for t in thetas])
det_fam = np.abs(np.cos(thetas / 2) * np.sin(thetas / 2))
print("\nfamily: S by hand == helper:", np.allclose(S_hand, S_lib, atol=1e-9))

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(thetas, S_hand, color=COLORS["quantum"], lw=2.5, label=r"entanglement entropy $S(\rho_A) = h(\cos^2\frac{\theta}{2})$ (bits)")
ax.plot(thetas, 2 * det_fam, color=COLORS["highlight"], lw=2, ls="--", label=r"$2\,|\det\Psi| = |\sin\theta|$  (01/17's product test)")
ax.plot(thetas, 1 - 2 * det_fam**2, color=COLORS["muted"], lw=1.5, ls=":", label=r"purity of a part $= 1 - 2|\det\Psi|^2$")
ax.axvline(np.pi / 2, color=COLORS["grid"], lw=1)
ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi]); ax.set_xticklabels(["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.set_xlabel(r"$\theta$  in  $\cos\frac{\theta}{2}|00\rangle + \sin\frac{\theta}{2}|11\rangle$"); ax.set_ylabel("value")
ax.set_title("From a product to the Bell state and back: three readings of one number", pad=12)
ax.legend(fontsize=9, loc="lower center")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The eigenvalue coin of $I/2$ is $(\tfrac12, \tfrac12)$ — one bit, by hand, by `von_neumann_entropy`, and by the course's `entanglement_entropy`; a product has $0$. On a random pure state the two parts have identical eigenvalues (so $S(\rho_A) = S(\rho_B)$), their sum is $1$ and their product is $|\det\Psi|^2$, as derived; a random local gate $U\otimes V$ does not move $S$ at all. On the family, the by-hand $h(\cos^2\tfrac\theta2)$ coincides with the helper along the whole curve: $0$ at the two product ends, $1$ bit at the Bell point $\theta = \pi/2$, where $|\det\Psi|$ peaks and the part's purity bottoms out at $\tfrac12$. Three curves, one quantity: how far from a product the pure whole is.

## 3. The honest limit: a mixed whole breaks the yardstick

Here is the trap, and it matters for the rest of the course. Take the correlated coins, $\rho_{AB} = \tfrac12|00\rangle\langle00| + \tfrac12|11\rangle\langle11|$. Their part is $I/2$ — *one full bit of "entanglement entropy"* — and they are **not entangled**: they are a coin shared by two parties, a mixture of the products $|00\rangle$ and $|11\rangle$. $S(\rho_A)$ measures entanglement **only when the whole is pure**; on a mixed whole it mixes up two things — the part's share of the whole's own randomness and any entanglement — and cannot tell them apart. When you meet an entropy of a part later in the course, the first question is always: *is the global state pure?*

So what *is* entanglement for a mixed state? The definition flips to the classical side: a mixed $\rho_{AB}$ is **separable** if it is a mixture of products, $\rho_{AB} = \sum_i p_i\,\rho_A^{(i)}\otimes\rho_B^{(i)}$ — a shared random recipe for independent preparations, which is exactly what a classical joint table is — and **entangled** otherwise. The coins are separable by construction.

For two qubits there is a clean, computable test (Peres 1996; Horodecki 1996 — necessary *and* sufficient in this dimension): the **partial transpose**. Transpose the B-indices only — in block language, transpose *each* $2\times2$ block $[\rho]_{aa'}$ in place, $(\rho^{T_B})_{ab,a'b'} = \rho_{ab',a'b}$. A separable state stays a valid density matrix under this (each $\rho_B^{(i)\mathsf T}$ is still a state), so all its eigenvalues stay $\ge 0$. **A negative eigenvalue proves entanglement.** For the Bell state, transposing the blocks moves the two corners $\tfrac12$ from $(00,11),(11,00)$ to $(01,10),(10,01)$, and the resulting matrix has eigenvalues $\tfrac12, \tfrac12, \tfrac12, -\tfrac12$. For the coins (diagonal), nothing moves: eigenvalues $\tfrac12, \tfrac12, 0, 0$.

In [ ]:
def partial_transpose_B(rho4):
    # transpose each 2x2 block in place: (rho^{T_B})_{ab, a'b'} = rho_{ab', a'b}
    out = np.zeros_like(rho4)
    for a in range(2):
        for a_ in range(2):
            out[2 * a: 2 * a + 2, 2 * a_: 2 * a_ + 2] = rho4[2 * a: 2 * a + 2, 2 * a_: 2 * a_ + 2].T
    return out

rho_pp = density_matrix(tensor(KET_PLUS, KET_PLUS))
print("the trap: S(rho_A) of the coins =", round(entanglement_entropy(coins, dims=[2, 2]), 3), "bit   purity of the whole =", purity(coins))
print("\npartial-transpose eigenvalues (negative = entangled):")
for name, rho in [("Bell", rho_bell), ("coins (01/16)", coins), ("|+>|+>", rho_pp), ("family theta = pi/4",
                  density_matrix(np.array([np.cos(np.pi / 8), 0, 0, np.sin(np.pi / 8)], dtype=complex)))]:
    ev = np.linalg.eigvalsh(partial_transpose_B(rho)).real
    ev = np.where(np.abs(ev) < 1e-9, 0.0, ev)              # clip numerical -0.
    print(f"  {name:20s} {np.round(ev, 3)}   entangled: {ev.min() < -1e-9}")
print("\nBell partial transpose (real part) -- the corners moved inward:\n", np.round(partial_transpose_B(rho_bell).real, 2))

**Read the output.** The trap first, in numbers: the coins carry $S(\rho_A) = 1$ bit with a whole of purity $\tfrac12$ — the entropy of the part says nothing here. Then the test that does work on mixtures: the Bell state's partial transpose has an eigenvalue $-\tfrac12$ (entangled), the coins' stays non-negative (separable, as built), the product $|{+}{+}\rangle$ stays non-negative (separable), and the tilted member of the family at $\theta = \pi/4$ is entangled but less so (its negative eigenvalue is smaller in magnitude — exercise 3 follows the whole curve). The printed matrix shows what transposing the blocks did to the Bell state: the corners moved from $|00\rangle\langle11|$ to $|01\rangle\langle10|$, where they sit on a zero diagonal — a $2\times2$ block $\begin{pmatrix}0&\tfrac12\\\tfrac12&0\end{pmatrix}$ whose eigenvalues are $\pm\tfrac12$. That minus sign is entanglement, certified.

## 4. Telling the Bell pair from the coins: the second basis

`01/17` ended on the uncomfortable fact that a $Z$-record cannot tell the Bell state from the coins — both give `'00'` and `'11'` half the time each, never `'01'` or `'10'`. What tells them apart is what `01/08` taught for one qubit: measure in **another basis**. Apply $H$ to both qubits before reading, i.e. measure $X\otimes X$.

By hand: $X\otimes X$ swaps $|00\rangle \leftrightarrow |11\rangle$ and $|01\rangle \leftrightarrow |10\rangle$, so $X\otimes X\,|\Phi^+\rangle = |\Phi^+\rangle$ — the Bell state is an eigenvector with eigenvalue $+1$: $\langle X\otimes X\rangle = 1$, the two $X$-readings *always agree*. For the coins, each branch is a product of $Z$-eigenstates, whose $X$-readings are independent fair coins: $\langle X\otimes X\rangle = 0$, agreement half the time. (And $\langle Y\otimes Y\rangle = -1$ for $|\Phi^+\rangle$, $0$ for the coins — exercise 1.) Both states have $\langle Z\otimes Z\rangle = 1$ and all single-qubit averages $0$: identical in $Z$, opposite in $X$.

Let's compute the three two-qubit correlators from the density matrices, then run the $X$-basis Bell circuit on the simulator — the experiment `01/17` could not yet do.

In [ ]:
corr = {"ZZ": tensor(PAULI_Z, PAULI_Z), "XX": tensor(PAULI_X, PAULI_X), "YY": tensor(PAULI_Y, PAULI_Y)}
print("X (x) X |Phi+> == |Phi+>:", np.allclose(corr["XX"] @ bell, bell))
print(f"{'':8s}" + "".join(f"{k:>8s}" for k in corr) + "     <X>_A  <Z>_A")
for name, rho in [("Bell", rho_bell), ("coins", coins), ("|+>|+>", density_matrix(tensor(KET_PLUS, KET_PLUS)))]:
    vals = [np.trace(rho @ M).real for M in corr.values()]
    loc = [np.trace(rho @ tensor(P, IDENTITY)).real for P in (PAULI_X, PAULI_Z)]
    print(f"{name:8s}" + "".join(f"{v:+8.2f}" for v in vals) + "    " + "".join(f"{v:+6.2f} " for v in loc))

# the experiment: Bell circuit (course convention, 01/17), then H on both qubits = read X (x) X
sim = AerSimulator()
N = 4096
qc = QuantumCircuit(2, 2)
qc.h(1); qc.cx(1, 0)            # Bell pair
qc.h(0); qc.h(1)                # rotate both to the X basis (01/08)
qc.measure([0, 1], [0, 1])
counts = sim.run(transpile(qc, sim), shots=N, seed_simulator=int(rng.integers(2**31))).result().get_counts()
agree = (counts.get("00", 0) + counts.get("11", 0)) / N
print(f"\nX-basis counts on the Bell pair: {dict(sorted(counts.items()))}   agreement = {agree:.4f}  (Born: 1; coins would give 0.5 +- {1/(2*np.sqrt(N)):.3f})")

**Read the output.** $X\otimes X$ leaves $|\Phi^+\rangle$ unchanged, so its average is $+1$; the coins give $0$. The table is the fingerprint: Bell and coins agree on $\langle Z\otimes Z\rangle = 1$ and on every single-qubit average ($0$, both parts $I/2$), and part company on $\langle X\otimes X\rangle$ ($1$ vs $0$) and $\langle Y\otimes Y\rangle$ ($-1$ vs $0$); the product $|{+}{+}\rangle$ has $\langle X\otimes X\rangle = 1$ too but $\langle Z\otimes Z\rangle = 0$ — no *single* correlator is the signature, the *pattern* across bases is. On the simulator, the Bell pair read in the $X$ basis again agrees on every shot — $4096$ out of $4096$ — where a shared coin would agree on half of them, give or take $0.008$. One more basis and the two states, indistinguishable in $Z$, are told apart with certainty. In fact the pair $(\langle Z\otimes Z\rangle, \langle X\otimes X\rangle) = (1, 1)$ already *certifies* entanglement on its own: for a product state $\langle Z\otimes Z\rangle + \langle X\otimes X\rangle = z_A z_B + x_A x_B \le |\vec r_A|\,|\vec r_B| \le 1$ (Cauchy–Schwarz on the Bloch vectors), and a mixture of products cannot exceed the bound either — so any separable state has $\langle Z\otimes Z\rangle + \langle X\otimes X\rangle \le 1$, and the Bell state's $2$ is out of reach. (The three same-basis correlators alone could still be mimicked by a cleverer shared recipe with *pre-agreed* answers for each basis; what no shared recipe reproduces is the table with *mismatched* bases — Bell's theorem, a story for a later course; the references point the way.)

## Your turn

1. **The other three.** Build $|\Phi^-\rangle = (I\otimes Z)|\Phi^+\rangle$, $|\Psi^+\rangle = (I\otimes X)|\Phi^+\rangle$ and $|\Psi^-\rangle = (I\otimes XZ)|\Phi^+\rangle$ — local gates on B alone. Predict, then check: their entanglement entropy (why must it be $1$ bit without computing?), their $\det\Psi$, and their table of $\langle Z\otimes Z\rangle, \langle X\otimes X\rangle, \langle Y\otimes Y\rangle$ (each is $\pm1$; the four sign patterns are distinct — the four states are mutually orthogonal and form a basis, the Bell basis).
2. **A product has none — and nothing else has none.** For $|\varphi\rangle\otimes|\chi\rangle$ show by hand that $\rho_A = |\varphi\rangle\langle\varphi|$ (use `01/16`'s product rule), hence eigenvalues $1, 0$ and $S = 0$; check on $|0\rangle\otimes|+\rangle$ and on a random product. Then the converse claimed in section 1: if a pure $|\Psi\rangle$ has a pure part, $\rho_A$ has eigenvalues $1, 0$, so $\lambda_1\lambda_2 = |\det\Psi|^2 = 0$ (section 2), so $\det\Psi = 0$ and the state is a product (`01/17`). Write it out.
3. **The whole family, three ways.** For $\theta \in [0, \pi]$ plot, on one axis, $S(\rho_A)$, the most negative eigenvalue of the partial transpose, and $\langle X\otimes X\rangle$ of $\cos\tfrac\theta2|00\rangle + \sin\tfrac\theta2|11\rangle$. Derive by hand that the negative eigenvalue is $-\tfrac12\sin\theta$ and that $\langle X\otimes X\rangle = \sin\theta$; at which $\theta$ are all three extremal?
4. **Harder: the trap, made sharp.** The Werner state $\rho_p = p\,|\Phi^+\rangle\langle\Phi^+| + (1-p)\,\tfrac{I}{4}$ ($p \in [0, 1]$: a Bell pair mixed with pure noise). Show by hand that $\rho_A = I/2$ for *every* $p$, so $S(\rho_A) = 1$ bit throughout — the entropy of the part is blind to $p$. Then compute the partial transpose's eigenvalues by hand (the corners $p/2$ move inward onto the $\tfrac{1-p}{4}$ diagonal) and show $\rho_p$ is entangled if and only if $p > \tfrac13$. Plot both quantities against $p$ and write one sentence on why, later in the course, one must check the purity of the whole before reading an entropy of a part.

## What you built

- You defined entanglement for pure states (not a product, $\det\Psi \ne 0$), took the Bell state's partial trace by blocks to $I/2$, and set the purity table — point mass / coins / Bell — that shows "pure whole, mixed parts" is the row no classical table has.
- You computed the entanglement entropy by hand from the eigenvalues of a part, proved $S(\rho_A) = S(\rho_B)$ and $\lambda_1\lambda_2 = |\det\Psi|^2$, checked invariance under local gates, and followed $h(\cos^2\tfrac\theta2)$ from a product to the Bell state and back.
- You met the trap — the coins have $S(\rho_A) = 1$ bit and no entanglement — fixed the rule *entropy of a part measures entanglement only for a pure whole*, defined separability for mixtures, and certified entanglement with the partial transpose ($-\tfrac12$ for Bell, nothing negative for the coins).
- You told the Bell pair from the coins with a second basis, $\langle X\otimes X\rangle = 1$ versus $0$, by hand and on the simulator.

Entanglement is now a definition, a number, a caveat and an experiment in your hands — the resource the rest of the course keeps returning to, read without over-reading. Genuinely well done.

**Dictionary so far** (the table from `01/17`, plus today's row):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| transport plan | quantum coupling |

(The last row is still forward-looking — earned in `04/04`.)

## What's next

You have built states (`01/12`), compared them (`01/14`), combined them (`01/16`), correlated them (`01/17`) and now entangled them. The last foundation is how states *change* when they are not alone — `01/19_quantum_channels`: completely positive trace-preserving maps, Kraus operators, and the fact `01/16` hinted at: a qubit decoheres *because* it becomes entangled with an environment you then trace out. Today's partial trace and today's $I/2$ are about to reappear as the endpoint of a noisy channel.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.4.3 (reduced density operator), §2.5 (Schmidt decomposition and purifications: $S(\rho_A) = S(\rho_B)$), §11.3 (von Neumann entropy), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- E. Schrödinger, *Discussion of probability relations between separated systems*, Math. Proc. Cambridge Phil. Soc. **31**, 555 (1935) — the word "entanglement" and the pure-whole/mixed-parts observation. doi:10.1017/S0305004100013554
- A. Einstein, B. Podolsky & N. Rosen, *Can quantum-mechanical description of physical reality be considered complete?*, Phys. Rev. **47**, 777 (1935). doi:10.1103/PhysRev.47.777
- J. S. Bell, *On the Einstein Podolsky Rosen paradox*, Physics **1**, 195 (1964) — why no shared classical recipe reproduces the correlations across mismatched bases. doi:10.1103/PhysicsPhysiqueFizika.1.195
- R. F. Werner, *Quantum states with Einstein-Podolsky-Rosen correlations admitting a hidden-variable model*, Phys. Rev. A **40**, 4277 (1989) — the definition of separable (mixture of products) for mixed states. doi:10.1103/PhysRevA.40.4277
- A. Peres, *Separability criterion for density matrices*, Phys. Rev. Lett. **77**, 1413 (1996) — the partial-transpose test. doi:10.1103/PhysRevLett.77.1413
- M. Horodecki, P. Horodecki & R. Horodecki, *Separability of mixed states: necessary and sufficient conditions*, Phys. Lett. A **223**, 1 (1996) — the test is complete for two qubits. doi:10.1016/S0375-9601(96)00706-2
- R. Horodecki, P. Horodecki, M. Horodecki & K. Horodecki, *Quantum entanglement*, Rev. Mod. Phys. **81**, 865 (2009) — the survey, including entanglement measures and why the entropy of a part is a measure for pure states only. doi:10.1103/RevModPhys.81.865

**Previous:** `notebooks/01_QuantumFoundations/17_cnot_and_two_qubit_gates.ipynb` · **Next:** `notebooks/01_QuantumFoundations/19_quantum_channels.ipynb`